# Phase 1 — Data Cleaning & Core Metrics

**Objective:** Load, clean, and compute the five core product metrics a Product Analyst tracks on Day 1.

**Dataset:** Online Retail II — UCI Machine Learning Repository  
**Analyst:** Sreya Ghosh  

---

### Business Questions
1. How is monthly revenue trending?
2. How many active customers do we have each month (MAU)?
3. What is the Average Order Value (AOV)?
4. How frequently do customers purchase?
5. Which markets drive the most revenue?

## 1. Imports & Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot styling
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

NAVY   = '#1A1A2E'
ACCENT = '#E94560'
GRAY   = '#555555'

print('Libraries loaded successfully')

## 2. Load Data

In [ ]:
# Load both years and combine
df1 = pd.read_excel('../data/online_retail_II.xlsx', sheet_name='Year 2009-2010')
df2 = pd.read_excel('../data/online_retail_II.xlsx', sheet_name='Year 2010-2011')
df_raw = pd.concat([df1, df2], ignore_index=True)

print(f'Raw dataset: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')
print(f'Date range: {df_raw["InvoiceDate"].min()} to {df_raw["InvoiceDate"].max()}')
df_raw.head()

## 3. Data Quality Assessment

In [ ]:
print('=== NULL VALUES ===')
print(df_raw.isnull().sum())
print(f'\nNull % for Customer ID: {df_raw["Customer ID"].isnull().mean()*100:.1f}%')

print('\n=== CANCELLATIONS ===')
cancellations = df_raw[df_raw['Invoice'].astype(str).str.startswith('C')]
print(f'Cancellation rows: {len(cancellations):,}')

print('\n=== NEGATIVE VALUES ===')
print(f'Negative quantity rows: {(df_raw["Quantity"] < 0).sum():,}')
print(f'Zero/negative price rows: {(df_raw["Price"] <= 0).sum():,}')

## 4. Data Cleaning

In [ ]:
df = df_raw.copy()

# Step 1: Drop missing Customer IDs (can't track behaviour without them)
df = df.dropna(subset=['Customer ID'])

# Step 2: Remove cancellations
df = df[~df['Invoice'].astype(str).str.startswith('C')]

# Step 3: Remove nonsensical values
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

# Step 4: Feature engineering
df['Revenue']     = df['Quantity'] * df['Price']
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Month']       = df['InvoiceDate'].dt.to_period('M')
df['Year']        = df['InvoiceDate'].dt.year
df['Customer ID'] = df['Customer ID'].astype(int)

# Cleaning summary
removed = df_raw.shape[0] - df.shape[0]
print(f'Rows removed: {removed:,} ({removed/df_raw.shape[0]*100:.1f}%)')
print(f'Clean dataset: {df.shape[0]:,} rows')
print(f'Unique customers: {df["Customer ID"].nunique():,}')
print(f'Unique products:  {df["StockCode"].nunique():,}')
print(f'Date range: {df["InvoiceDate"].min().date()} to {df["InvoiceDate"].max().date()}')

## 5. Core Metrics

In [ ]:
# --- Metric 1 & 2: Monthly Revenue and MAU ---
monthly = df.groupby('Month').agg(
    Revenue=('Revenue', 'sum'),
    MAU=('Customer ID', 'nunique'),
    Orders=('Invoice', 'nunique')
).reset_index()
monthly['Month_str'] = monthly['Month'].astype(str)

# --- Metric 3: AOV (Average Order Value) ---
order_totals = df.groupby('Invoice')['Revenue'].sum()
aov = order_totals.mean()

# --- Metric 4: Purchase Frequency ---
purchase_freq = df.groupby('Customer ID')['Invoice'].nunique()

# --- Metric 5: Revenue by Country ---
country_revenue = df.groupby('Country')['Revenue'].sum().sort_values(ascending=False)

print('===== CORE METRICS SUMMARY =====')
print(f'Total Revenue:          £{df["Revenue"].sum():>12,.0f}')
print(f'Average Order Value:    £{aov:>12,.2f}')
print(f'Avg Orders/Customer:    {purchase_freq.mean():>12.1f}')
print(f'Avg Monthly Revenue:    £{monthly["Revenue"].mean():>12,.0f}')
print(f'Avg MAU:                {monthly["MAU"].mean():>12.0f}')

## 6. Visualisations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('E-Commerce Core Metrics Dashboard', fontsize=16, fontweight='bold', color=NAVY, y=1.01)

months = monthly['Month_str']
tick_every = 3  # show every 3rd label so x-axis isn't crowded

# Chart 1 — Monthly Revenue
ax = axes[0, 0]
ax.plot(months, monthly['Revenue'], color=ACCENT, linewidth=2.5, marker='o', markersize=4)
ax.fill_between(range(len(months)), monthly['Revenue'], alpha=0.1, color=ACCENT)
ax.set_title('Monthly Revenue (£)', fontweight='bold', color=NAVY)
ax.set_xticks(range(0, len(months), tick_every))
ax.set_xticklabels([months.iloc[i] for i in range(0, len(months), tick_every)], rotation=45, ha='right')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}K'))
ax.grid(axis='y', alpha=0.3)

# Chart 2 — MAU
ax = axes[0, 1]
ax.bar(months, monthly['MAU'], color=NAVY, alpha=0.85)
ax.set_title('Monthly Active Users (MAU)', fontweight='bold', color=NAVY)
ax.set_xticks(range(0, len(months), tick_every))
ax.set_xticklabels([months.iloc[i] for i in range(0, len(months), tick_every)], rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3)

# Chart 3 — Purchase Frequency Distribution
ax = axes[1, 0]
freq_capped = purchase_freq.clip(upper=20)  # cap at 20 for readability
ax.hist(freq_capped, bins=20, color=ACCENT, alpha=0.8, edgecolor='white')
ax.set_title('Purchase Frequency Distribution\n(orders per customer, capped at 20)', fontweight='bold', color=NAVY)
ax.set_xlabel('Number of Orders')
ax.set_ylabel('Number of Customers')
ax.grid(axis='y', alpha=0.3)
ax.axvline(purchase_freq.mean(), color=NAVY, linestyle='--', label=f'Mean: {purchase_freq.mean():.1f}')
ax.legend()

# Chart 4 — Top 10 Countries by Revenue (excl. UK which dominates)
ax = axes[1, 1]
top_non_uk = country_revenue[country_revenue.index != 'United Kingdom'].head(10)
ax.barh(top_non_uk.index[::-1], top_non_uk.values[::-1], color=NAVY, alpha=0.85)
ax.set_title('Top 10 Countries by Revenue\n(excl. United Kingdom)', fontweight='bold', color=NAVY)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x/1000:.0f}K'))
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/phase1_core_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to outputs/phase1_core_metrics.png')

## 7. Analyst Notes — What I'm Seeing

> **This is the most important cell in the notebook.** A data dump without interpretation is reporting. Interpretation is analysis.

*(Fill this in after your charts render — write what you actually notice)*

**Revenue trend:**  
- [ ] Is there consistent MoM growth?
- [ ] Is there a seasonal spike? When?
- [ ] Is there a month where revenue dropped unexpectedly?

**MAU trend:**  
- [ ] Does MAU follow revenue, or do they diverge?
- [ ] If revenue grows but MAU is flat — what does that tell you?

**Purchase frequency:**  
- [ ] Are most customers one-time buyers or repeat buyers?
- [ ] What % of customers bought more than 3 times?

**Geography:**  
- [ ] Which countries outside UK are significant?
- [ ] Is there an underserved market with high potential?

---
**My anomaly to investigate in Phase 3 (RCA):**  
*(Write one thing that surprised you here — this becomes your RCA topic)*